# Film collection: 1970-2019

Read the TMDB candidates from `Film_Selection.ipynb`, fetch their Cinemetrics HTML pages, check identity and measurement scope, then save.

The student chose TMDB rating >= 7.0 and vote count >= 1,000. The same rules apply to all five decades. The current pool uses the first vote-ranked page per decade. The student also chose a measured-duration / TMDB-runtime band of 0.90-1.10 for initial screening. Final sample size is not fixed.

Run top to bottom from the project root. Saved responses are used by default. Directory links were checked separately by exact title and year; all lookup responses and queries are kept in `01_raw/candidate_links`. These are candidate links, not proof of film identity or full-film coverage.

In [2]:
from pathlib import Path
from datetime import datetime, timezone, timedelta
import requests
from bs4 import BeautifulSoup
import pandas as pd
import json
import re
import time
import hashlib
import math

folder = Path.cwd()
live = False
duration_low, duration_high = 0.90, 1.10
base = folder / "01_raw/film_measurements"
latest = base / "latest_run.txt"
run_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ") if live else latest.read_text().strip()
run = base / run_id
run.mkdir(parents=True, exist_ok=True)
headers = {"User-Agent": "ADS206-film-study/0.2 (educational, limited requests)"}
proxy_url = None
if live:
    pool = json.loads((folder / "02_audit/tmdb_candidates.json").read_text(encoding="utf-8"))
    links_base = folder / "01_raw/candidate_links"
    links_id = (links_base / "latest_run.txt").read_text().strip()
    links = json.loads((links_base / links_id / "links.json").read_text(encoding="utf-8"))
    assert links["candidate_run"] == pool["run_id"], "Candidate and directory runs differ."
    pages, seen, link_review = [], set(), []
    for group in links["decades"]:
        for page in group["rows"]:
            matches = [film for film in pool["records"] if film["title"].strip().casefold() == str(page["title"]).strip().casefold()
                       and int(film["release_date"][:4]) == int(page["year"])]
            if len(matches) != 1:
                link_review.append({**page, "reason": "No unique candidate with the same title and year"})
            if len(matches) == 1 and page["uuid"] not in seen:
                pages.append({"page_id": page["uuid"], "tmdb_id": matches[0]["tmdb_id"],
                              "url": "https://cinemetrics.uchicago.edu/movie/" + page["uuid"],
                              "directory_director": page.get("director") if isinstance(page.get("director"), str) else ""})
                seen.add(page["uuid"])
    inputs = {"candidate_run": pool["run_id"], "links_run": links_id, "films": pool["records"], "pages": pages, "link_review": link_review,
              "rules": {key: pool[key] for key in ["date_start", "date_end", "rating_cutoff", "vote_count_cutoff"]}}
    (run / "inputs.json").write_text(json.dumps(inputs, ensure_ascii=False, indent=2), encoding="utf-8")
    for line in (folder / ".env").read_text(encoding="utf-8-sig").splitlines():
        if line.startswith("HTTPS_PROXY="):
            proxy_url = line.split("=", 1)[1].strip()
else:
    inputs = json.loads((run / "inputs.json").read_text(encoding="utf-8"))
films = {film["tmdb_id"]: film for film in inputs["films"]}
print("Mode:", "live requests" if live else "saved responses")
print("Candidate films:", len(films), "Unique measurement pages:", len(inputs["pages"]))

Mode: saved responses
Candidate films: 100 Unique measurement pages: 253


## 1. Request and save

`requests.get` retrieves HTML. Save each received response, its hash and a structured log. Wait between requests and retry temporary failures up to three times. Missing responses stay marked as unavailable.

In [3]:
def fetch(page):
    name = page["page_id"] + ".html"
    if not live:
        return (run / name).read_bytes() if (run / name).exists() else None
    for attempt in range(3):
        reply = None
        clock = time.perf_counter()
        log = {"time_utc": datetime.now(timezone.utc).isoformat(), "url": page["url"],
               "retry_count": attempt, "collector": "AI-assisted HTML acquisition"}
        try:
            reply = requests.get(page["url"], headers=headers, timeout=(10, 30),
                                 proxies={"https": proxy_url} if proxy_url else None)
            tag = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
            file = f"{page['page_id']}_{tag}.html"
            (run / file).write_bytes(reply.content)
            log.update(status=reply.status_code, file=file, url=reply.url,
                       content_type=reply.headers.get("Content-Type"),
                       redirects=[r.status_code for r in reply.history],
                       retry_after=reply.headers.get("Retry-After"),
                       sha256=hashlib.sha256(reply.content).hexdigest())
        except requests.RequestException as error:
            log["error"] = type(error).__name__
        log["seconds"] = round(time.perf_counter() - clock, 3)
        with (run / "requests.jsonl").open("a", encoding="utf-8") as file:
            file.write(json.dumps(log) + "\n")
        time.sleep(1.2)
        if reply is not None and reply.ok:
            (run / name).write_bytes(reply.content)
            return reply.content
        if reply is not None and reply.status_code != 429 and reply.status_code < 500:
            return None
        if attempt == 2:
            return None
        delay = 2 ** attempt
        if reply is not None and reply.headers.get("Retry-After"):
            wait = reply.headers["Retry-After"]
            if not wait.isdigit() or int(wait) > 60:
                raise RuntimeError("Stop and review Retry-After before more requests.")
            delay = max(delay, int(wait))
        time.sleep(delay)

## 2. Parse and check each measurement

Read the header, statistic buttons, IMDb links and comment. Each overall statistic button has two direct spans; checking labels and duplicate fields helps detect layout changes.

Use shared IMDb ID and year where available. Otherwise require exact title, year and a shared director name from the directory and TMDB credits. Uncertain cases stay pending; no fuzzy matching is used.

Convert the measured duration to seconds. The student's 0.90-1.10 runtime band is a screening rule, not proof of a whole film. Clear partial-measurement wording, impossible numeric fields and source accuracy warnings also block initial acceptance. Generic mentions of a scene can describe a whole-film breakdown and are not enough by themselves to reject a record.

In [4]:
def seconds(value):
    parts = value.split(":")
    return sum(float(part) * 60 ** index for index, part in enumerate(reversed(parts)))

records, errors, chosen = [], [], {}
for number, page in enumerate(inputs["pages"], 1):
    raw = fetch(page)
    if raw is None:
        errors.append({**page, "error": "response unavailable"})
        continue
    try:
        soup = BeautifulSoup(raw, "html.parser")
        heading = soup.find("header").get_text(" ", strip=True)
        title = re.search(r"^(.*?)\s*\(\s*(\d{4})", heading)
        assert title, "Check the page heading."
        stats = {}
        for button in soup.select("button"):
            parts = button.find_all("span", recursive=False)
            if len(parts) == 2:
                label = parts[0].get_text(strip=True).rstrip(":")
                if label in {"NoS", "LEN", "ASL", "MSL"}:
                    assert label not in stats, "Repeated overall field."
                    stats[label] = parts[1].get_text(strip=True)
        assert len(stats) == 4, "Check the overall shot fields."
        ids = {re.search(r"tt\d+", a["href"]).group()
               for a in soup.find_all("a", href=re.compile(r"imdb\.com/title/tt\d+"))}
        comment = re.search(r"comment:\s*(.*?)\s+NoS\s*:", soup.get_text(" ", strip=True), re.I)
        film = films[page["tmdb_id"]]
        year = int(title[2])
        year_ok = year == int(film["release_date"][:4])
        names = [name.strip().casefold() for name in re.split(r",|;|/|&|\band\b", page["directory_director"])]
        director_ok = any(name.casefold() in names for name in film.get("directors", []))
        title_ok = title[1].strip().casefold() == film["title"].strip().casefold()
        identity = "IMDb + year" if film.get("imdb_id") in ids and year_ok else (
            "title + year + director" if title_ok and year_ok and director_ok else "review")
        record = {**film, **page, "title_html": title[1].strip(), "year_html": year,
            "imdb_ids_html": sorted(ids), "shot_count": int(stats["NoS"].replace(",", "")),
            "measured_duration": stats["LEN"], "asl_seconds": float(stats["ASL"]),
            "msl_seconds": float(stats["MSL"]), "comment": comment[1] if comment else "",
            "identity_check": identity,
            "measurement_check": "review boundaries", "selected_for_main_analysis": False}
        record["measured_seconds"] = seconds(record["measured_duration"])
        record["duration_ratio"] = record["measured_seconds"] / (film["runtime_minutes"] * 60) if film.get("runtime_minutes", 0) > 0 else None
        wording = record["title_html"] + " " + record["comment"]
        partial = re.search(r"\b(first|last)\s+(\d+|hour|half)\b|\bsecond half\b|\bexcerpt\b|\bclip\b|\btrailer\b|scene only|volleyball scene", wording, re.I)
        record["measurement_check"] = "partial wording" if partial else "review boundaries"
        record["accuracy_warning"] = bool(re.search(r"not (very )?accurate|inaccurate|rough estimate", record["comment"], re.I))
        record["edition_check"] = "source edition not independently verified"
        record["numeric_check"] = all(math.isfinite(record[key]) and record[key] > 0
            for key in ["shot_count", "asl_seconds", "msl_seconds", "measured_seconds"])
        record.pop("cinemetrics_check", None)
        reasons = []
        if record["identity_check"] == "review": reasons.append("identity review")
        if record["duration_ratio"] is None or not duration_low <= record["duration_ratio"] <= duration_high:
            reasons.append("outside duration band")
        if record["measurement_check"] == "partial wording": reasons.append("partial wording")
        if not record["numeric_check"]: reasons.append("numeric review")
        if record["accuracy_warning"]: reasons.append("source accuracy warning")
        record["review_reasons"] = reasons
        record["passes_initial_checks"] = not reasons
        if not reasons:
            uid = record["tmdb_id"]
            key = (abs(record["duration_ratio"] - 1), record["page_id"])
            if uid not in chosen or key < (abs(chosen[uid]["duration_ratio"] - 1), chosen[uid]["page_id"]):
                chosen[uid] = record
        records.append(record)
    except (AssertionError, AttributeError, KeyError, TypeError, ValueError) as error:
        errors.append({**page, "error": type(error).__name__})
    if number % 20 == 0:
        print(f"Pages checked: {number}/{len(inputs['pages'])}; passing films so far: {len(chosen)}", flush=True)
print("Parsed measurements:", len(records), "Pages needing parsing/request review:", len(errors))

Pages checked: 20/253; passing films so far: 1


Pages checked: 40/253; passing films so far: 11


Pages checked: 60/253; passing films so far: 18


Pages checked: 80/253; passing films so far: 24


Pages checked: 100/253; passing films so far: 26


Pages checked: 120/253; passing films so far: 27


... (3 outputs left out)

Pages checked: 200/253; passing films so far: 43


Pages checked: 220/253; passing films so far: 48


Pages checked: 240/253; passing films so far: 52


Parsed measurements: 253 Pages needing parsing/request review: 0


## 3. Build a working film sample

Accept a measurement only when identity is supported, the duration ratio is 0.90-1.10, numeric fields are positive/finite, and no clear partial wording or accuracy warning is present. A passing record is a full-film candidate under these recorded checks; edition/credits/playback differences remain a limitation.

For one film with several passing measurements, use the one closest to the reference runtime. Break a tie by page ID for reproducibility. Keep every alternative and rejection reason. This choice uses scope and identity, not ASL or the direction of a decade trend. A future live collector should count passing films and continue down the same vote-ranked candidate order when needed.

Checks run immediately after each page is parsed. Rejected pages are evidence, not accepted film records. Passing films are counted during the loop; a final whole-dataset audit is still required.

In [5]:
for row in chosen.values(): row["selected_for_main_analysis"] = True
sample = list(chosen.values())
result = {"stage": "working film sample after recorded acquisition checks; edition not independently verified",
          "run_id": run_id, "candidate_run": inputs["candidate_run"], "links_run": inputs["links_run"],
          "candidate_films": len(films), "requested_unique_pages": len(inputs["pages"]),
          "selection_rules": {**inputs["rules"], "duration_low": duration_low, "duration_high": duration_high},
          "working_film_sample_size": len(sample), "final_sample_locked": False, "link_review": inputs["link_review"],
          "collector": "AI-assisted acquisition; rules confirmed by student", "records": records, "errors": errors}
if live:
    body = json.dumps(result, ensure_ascii=False, indent=2)
    (run / "records.json").write_text(body, encoding="utf-8")
    (folder / "02_audit/film_measurements.json").write_text(body, encoding="utf-8")
    latest.write_text(run_id, encoding="utf-8")
print("Parsed measurements:", len(records), "Unique films:", len({row["tmdb_id"] for row in records}))
print("Working sample: one passing measurement per film:", len(sample))
pd.DataFrame(sample)[["decade", "title", "measured_duration", "runtime_minutes", "identity_check"]]

Parsed measurements: 253 Unique films: 80
Working sample: one passing measurement per film: 54


,decade,title,measured_duration,runtime_minutes,identity_check
0,1970s,Alien,01:53:48.1,117,IMDb + year
1,1970s,The Exorcist,01:57:20.8,122,IMDb + year
2,1970s,Robin Hood,01:19:09.3,83,IMDb + year
3,1970s,Halloween,01:28:18.4,91,IMDb + year
4,1970s,Rocky II,01:55:34.4,119,IMDb + year
5,1970s,Rocky,01:56:31.3,119,IMDb + year
6,1970s,Jaws,02:01:17.6,124,IMDb + year
7,1970s,Monty Python and the Holy Grail,01:25:28.9,91,IMDb + year
8,1970s,Carrie,01:32:26.5,98,IMDb + year
9,1970s,The Godfather Part II,03:07:15.1,202,IMDb + year


## 4. Data passport

Record the active collection window, request outcomes, units, rules and field dictionary. Main REST requests, HTML requests and earlier aborted trials are reported separately. This is collection evidence; the formal missingness, outlier and downstream-impact analyses still belong in the analysis notebook.

In [6]:
sources, times = {}, []
for channel, path in {"REST_API": folder / "01_raw/tmdb_candidates" / inputs["candidate_run"], "HTML": run}.items():
    logs = [json.loads(line) for line in (path / "requests.jsonl").read_text().splitlines()]
    good = sum(200 <= row.get("status", 0) < 300 for row in logs)
    failures = {}
    for row in logs:
        times.append((datetime.fromisoformat(row["time_utc"]), row["seconds"]))
        if not 200 <= row.get("status", 0) < 300:
            cause = row.get("error", str(row.get("status")))
            failures[cause] = failures.get(cause, 0) + 1
    sources[channel] = {"run": path.relative_to(folder).as_posix(), "attempts": len(logs),
        "successful_responses": good, "success_rate": good / len(logs), "failures": failures,
        "received_response_bytes": sum((path / row["file"]).stat().st_size for row in logs if row.get("file"))}
fields = {}
derived_fields = {"decade", "page", "rank_on_page", "tmdb_url", "selected_for_main_analysis", "rating_and_votes_check", "cinemetrics_check"}
api_fields = set(inputs["films"][0])
html_fields = {"title_html", "year_html", "imdb_ids_html", "shot_count", "measured_duration", "asl_seconds", "msl_seconds", "comment"}
for name in records[0]:
    values = [row.get(name) for row in records]
    present = [value for value in values if value is not None] if name == "review_reasons" else [value for value in values if value not in (None, "", [], {})]
    example = present[0] if present else None
    fields[name] = {"source": "HTML" if name in html_fields else ("derived" if name in derived_fields else ("REST_API" if name in api_fields else "directory or derived")),
        "type": type(example).__name__, "example": example[:120] if isinstance(example, str) else example,
        "missing_rate": 1 - len(present) / len(values)}
passport = {"status": result["stage"], "rules": result["selection_rules"], "main_runs": sources,
    "collection_start_utc": min(start for start, elapsed in times).isoformat(),
    "collection_end_utc": max(start + timedelta(seconds=elapsed) for start, elapsed in times).isoformat(),
    "measurement_records": len(records), "distinct_films_with_measurements": len({row["tmdb_id"] for row in records}),
    "working_film_records": len(sample), "candidate_films": len(films), "field_dictionary": fields,
    "field_dictionary_unit": "Parsed measurement records, before one-per-film selection",
    "units": {"asl_seconds": "seconds", "msl_seconds": "seconds", "measured_seconds": "seconds", "runtime_minutes": "minutes"},
    "limits": ["One vote-ranked candidate page per decade; not random", "Edition and playback speed not independently verified", "Below the written 2000-record floor; no scale compliance claim"],
    "log_scope": "Active REST and HTML runs only; directory/policy lookups, probes and earlier trials remain separately logged"}
if live:
    (folder / "data_passport.json").write_text(json.dumps(passport, ensure_ascii=False, indent=2), encoding="utf-8")
print("Passport prepared; fields:", len(fields))

Passport prepared; fields: 47


## Sources

HTML: [Cinemetrics](https://cinemetrics.uchicago.edu/). API metadata and TMDB credit are in `Film_Selection.ipynb`. Saved raw responses, selectors and request logs support replay and source checks.